In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

bookings_raw = pd.read_csv("hotel_bookings_final_project.csv")

bookings_raw.info()

bookings_raw.describe()

In [ ]:
bookings_raw.head()

In [ ]:
print("Duplicate rows:", bookings_raw.duplicated().sum())

print("\nMissing values:")
print(bookings_raw.isna().sum())

print("\nRegion labels:", sorted(bookings_raw["hotel"].dropna().unique()))

print("product_category labels:", sorted(bookings_raw["arrival_date_month"].dropna().unique()))

print("customer_type labels:", sorted(bookings_raw["country"].dropna().unique()))

print("returned labels:", sorted(bookings_raw["market_segment"].dropna().unique()))

print("returned labels:", sorted(bookings_raw["deposit_type"].dropna().unique()))

print("returned labels:", sorted(bookings_raw["customer_type"].dropna().unique()))

print("returned labels:", sorted(bookings_raw["arrival_date_month"].dropna().unique()))

invalid_cancelations = bookings_raw.loc[
    bookings_raw["is_canceled"].notna() &
    ~bookings_raw["is_canceled"].isin([0, 1])
]

invalid_lead_times = bookings_raw.loc[
    bookings_raw["lead_time"].notna() &
    ~bookings_raw["lead_time"] < 0
]

invalid_arrival_date_day_of_month = bookings_raw.loc[
    bookings_raw["arrival_date_day_of_month"].notna() &
    ~bookings_raw["arrival_date_day_of_month"].between(1, 31)
]

invalid_stays_in_weekend_nights = bookings_raw.loc[
    bookings_raw["stays_in_weekend_nights"].notna() &
    ~bookings_raw["stays_in_weekend_nights"].between(0, 2)
]

invalid_stays_in_week_nights = bookings_raw.loc[
    bookings_raw["stays_in_week_nights"].notna() &
    ~bookings_raw["stays_in_week_nights"].between(0, 5)
]

invalid_repeated_guests = bookings_raw.loc[
    bookings_raw["is_repeated_guest"].notna() &
    ~bookings_raw["is_repeated_guest"].isin([0, 1])
]

invalid_cancelations["is_canceled"]
invalid_lead_times["lead_time"]
invalid_arrival_date_day_of_month["arrival_date_day_of_month"]
invalid_stays_in_weekend_nights["stays_in_weekend_nights"]
invalid_stays_in_week_nights["stays_in_week_nights"]
invalid_repeated_guests["is_repeated_guest"]



Most of the data values are present and the data values seem to be clean. There are only a couple "country" values missing. Since they would not have a big affect on the result, we can simply fill the values with "unknown".
Although, there are 26 duplicate rows, it does not seem like it will couse any problems, because none of the data values need to be unique.

In [ ]:
bookings = bookings_raw

bookings["country"] = bookings["country"].fillna("Unknown")

print("\nMissing values:")
print(bookings.isna().sum())

# Ruhan's Part 👇


In [ ]:
bookings_b = bookings.copy()

### Total Staying Nights

In [ ]:

bookings_b["total_nights"] = (
    bookings_b["stays_in_weekend_nights"]
    + bookings_b["stays_in_week_nights"]
)

bookings_b[[
    "stays_in_weekend_nights",
    "stays_in_week_nights",
    "total_nights"
]].head()

---------------------------------------------------------------------------------------------------------

### Total Guests

In [ ]:
bookings_b["total_guests"] = (
    bookings_b["adults"]
    + bookings_b["children"]
    + bookings_b["babies"]
)

bookings_b[["adults", "children", "babies", "total_guests"]].head()

---------------------------------------------------------------------------------------------------------

### How much does each booking cost?

In [ ]:
bookings_b["estimated_booking_value"] = (
    bookings_b["average_daily_rate"]
    * bookings_b["total_nights"]
)

bookings_b[[
    "average_daily_rate",
    "total_nights",
    "estimated_booking_value"
]].head()

---------------------------------------------------------------------------------------------------------

### Guest arrival dates

In [ ]:
bookings_b["arrival_date"] = pd.to_datetime(
    bookings_b["arrival_date_year"].astype("string") + "-"
    + bookings_b["arrival_date_month"] + "-"
    + bookings_b["arrival_date_day_of_month"].astype("string"),
    format="%Y-%B-%d",
    errors="coerce"
)

bookings_b[[
    "arrival_date_year",
    "arrival_date_month",
    "arrival_date_day_of_month",
    "arrival_date"
]].head()

---------------------------------------------------------------------------------------------------------

## Summary

In [ ]:
bookings_b["total_nights"].describe().round(2)

##### zero-night bookings

In [ ]:
zero_nights = bookings_b.loc[
    bookings_b["total_nights"] == 0
]

print("Zero-night bookings:", len(zero_nights))
zero_nights.head()

##### stay analysis

In [ ]:
overnight_bookings = bookings_b.loc[
    bookings_b["total_nights"] > 0
].copy()

overnight_bookings["total_nights"].describe().round(2)

##### Stay length distribution

In [ ]:
fig, ax = plt.subplots()

ax.hist(
    overnight_bookings["total_nights"],
    bins=range(1, 30),
    color="#8b5cf6",
    edgecolor="white"
)

ax.set_title(f"Booked Stay Length (n={len(overnight_bookings)})")
ax.set_xlabel("Total nights")
ax.set_ylabel("Number of bookings")

plt.show()

##### **What can we observe?**

Most bookings are for 1–4 nights, while longer bookings are less common.
The average booked stay is 3.47 nights and the median is 3 nights.
Five zero-night bookings were excluded. This chart includes both
cancelled and non-cancelled bookings.

_________________________________________________________________________________________________________

### Checking the calculations 👍

In [ ]:
print("Total nights correct:", (
    bookings_b["total_nights"] ==
    bookings_b["stays_in_weekend_nights"] + bookings_b["stays_in_week_nights"]
).all())

print("Total guests correct:", (
    bookings_b["total_guests"] ==
    bookings_b["adults"] + bookings_b["children"] + bookings_b["babies"]
).all())

print("Booking values correct:", (
    bookings_b["estimated_booking_value"] ==
    bookings_b["average_daily_rate"] * bookings_b["total_nights"]
).all())

print("Missing or invalid arrival dates:",
      bookings_b["arrival_date"].isna().sum())

---------------------------------------------------------------------------------------------------------

# Exporting the Results 

In [ ]:

bookings_b.to_csv("hotel_bookings_cleaned.csv", index=False)

stay_summary = (
    overnight_bookings["total_nights"]
    .describe()
    .round(2)
    .reset_index()
)

stay_summary.columns = ["statistic", "total_nights"]

stay_summary.to_csv("hotel_booking_summary.csv", index=False)

## Verifying Exports

In [ ]:
loaded_bookings = pd.read_csv(
    "hotel_bookings_cleaned.csv",
    parse_dates=["arrival_date"]
)
loaded_summary = pd.read_csv("hotel_booking_summary.csv")

print("Dataset shape matches:", loaded_bookings.shape == bookings_b.shape)
print("Dataset columns match:",
      loaded_bookings.columns.tolist() == bookings_b.columns.tolist())

print("Summary shape matches:", loaded_summary.shape == stay_summary.shape)
print("Summary columns match:",
      loaded_summary.columns.tolist() == stay_summary.columns.tolist())

loaded_summary

---------------------------------------------------------------------------------------------------------

## Member C: Cancellation analysis - Subhanjan's Part

**Research question:** Do cancellation rates differ between the city and resort hotels?

We will summarize cancellation counts and percentage, then compare cancellation rates within each hotel.

In [ ]:
bookings_c = bookings_b.copy()

print("Bookings available for cancellation analysis:", len(bookings_c))

bookings_c[["hotel", "is_canceled"]].head(10)

In [ ]:
# for number of bookings with recorded cancellation status
valid_n = bookings["is_canceled"].count()

# status count, where not canceled = 0 and canceled = 1
counts = bookings["is_canceled"].value_counts().sort_index()

# combining counts and percnetages into one table
cancelation_summary = pd.DataFrame({
    "Count": counts,
    "Percentage": counts / valid_n * 100
})

cancelation_summary.index = ["Not canceled", "Canceled"]
cancelation_summary.index.name = "Cancelation status"

print(f"Valid sample size (n): {valid_n}")
cancelation_summary.round(2)

In [ ]:
# cancelation status within each hotel
hotel_cancelation_counts = pd.crosstab(
    bookings["hotel"],
    bookings["is_canceled"]
)

# replacing numeric code with readable labels
hotel_cancelation_counts = hotel_cancelation_counts.rename(
    columns={0: "Not canceled", 1: "Canceled"}
)

hotel_cancelation_counts

In [ ]:
hotel_cancelation_percentages = pd.crosstab(
    bookings["hotel"],
    bookings["is_canceled"],
    normalize="index"
) * 100

hotel_cancelation_percentages = hotel_cancelation_percentages.rename(
    columns={0: "Not canceled", 1: "Canceled"}
)

hotel_cancelation_percentages.round(2)

In [ ]:
import matplotlib.pyplot as plt

# cancelation percentage for each hotel
cancelation_rates = hotel_cancelation_percentages["Canceled"]

hotel_totals = hotel_cancelation_counts.sum(axis=1)

ax = cancelation_rates.plot(
    kind="bar",
    color=["steelblue", "teal"],
    figsize=(8,5),
    rot=0
)
ax.set_title("Cancelation rates by hotel")
ax.set_xlabel("Hotel type")
ax.set_ylabel("Bookings canceled (%)")
ax.set_ylim(0, 100)

for position, hotel in enumerate(cancelation_rates.index):
    rate = cancelation_rates.loc[hotel]
    n = hotel_totals.loc[hotel]

    ax.text(
        position,
        rate+2,
        f"{rate:.2f}%\n(n = {n})",
        ha="center"
    )

plt.tight_layout()
plt.savefig("cancelation_rates_by_hotel.png", dpi=300, bbox_inches="tight")
plt.show()

### What can we observe?

City Hotel had a cancellation rate of 43.46% (279 of 642 bookings), compared with 24.30% for Resort Hotel (87 of 358 bookings). The city hotel's rate was therefore **19.16 percentage points higher**.

Each percentage uses the total bookings within that hotel as its denominator, allowing comparison despite the different sample sizes. These results show an association between hotel type and cancellation in this sample, but they do not establish that hotel type causes cancellations.

### Limitations

- The dataset is a 1,000-record sample from two specific hotels in Portugal. These results may not represent other city or resort hotels.
- This comparison does not account for differences in lead time, deposit type, market segment, or season. These factors could help explain the cancellation-rate gap.
- Exact duplicate rows were retained under our current cleaning decision. If some represent accidental duplicate records, the cancellation rates could be affected.

### Recommendation

Hotel management should prioritize investigating cancellation patterns at City Hotel, which had the higher cancellation rate in this sample. Comparing cancellations by lead time and deposit type could help identify patterns worth investigating before changing booking policies.

### Next question

What reasons do guests give for canceling their bookings? Answering this would require cancellation-reason data, which is not included in the supplied dataset.

In [ ]:
# hotel table includes every valid booking
assert hotel_cancelation_counts.to_numpy().sum() == valid_n

# checking if the cancelation matches the overall total
assert (
    hotel_cancelation_counts["Canceled"].sum()
    == bookings["is_canceled"].eq(1).sum()
)

# check that percentages within each hotel add up to 100%
assert(
    hotel_cancelation_percentages.sum(axis=1).round(8) == 100
).all()

print("All checks passed: booking totals and percentages are consistent.")

In [ ]:
bookings_no_duplicates = bookings.drop_duplicates()

# percentages after removing exact duplicate rows
rates_no_duplicates = pd.crosstab(
    bookings_no_duplicates["hotel"],
    bookings_no_duplicates["is_canceled"],
    normalize="index"
)[1] * 100

# comparison
duplicate_comparison = pd.DataFrame({
    "All rows: cancelation (%)": hotel_cancelation_percentages["Canceled"],
    "Duplicates removed: cancelation (%)": rates_no_duplicates
})

print(f"Rows removed in comparison: {len(bookings) - len(bookings_no_duplicates)}")
duplicate_comparison.round(2)

### Sensitivity check: exact duplicates

To assess whether our cancellation finding depends on retaining exact duplicates, we repeated the comparison with 26 duplicate rows removed from a separate copy.

City Hotel’s cancellation rate changed from 43.46% to 42.23%, while Resort Hotel’s changed from 24.30% to 23.88%. The conclusion remained the same: City Hotel had the higher cancellation rate.

This check does not establish whether the duplicate rows represent errors or separate bookings with identical recorded values. Our main analysis retains all rows, following the current group cleaning decision.